In [43]:
#import numpy and pysindy. If running locally pysindy may need to be installed separately using pip
import numpy as np
import pysindy as ps

### The Henon-Heiles Hamiltonian

$$
H = \frac{1}{2}(p_1^2 + p_2^2) + \frac{1}{2}(q_1^2 + q_2^2) + \alpha q_1^2 q_2 - \frac{\beta}{3} q_2^3
$$

### 'True' Equations of motion 

$$
\begin{aligned}
\dot{q}_1 &= p_1 \\
\dot{q}_2 &= p_2 \\ 
\dot{p}_1 &= -q_1 -2\alpha q_1 q_2 \\
\dot{p}_2 &= -q_2 -\alpha q_1^2 + \beta q_2^2
\end{aligned}
$$

### How to run

- In order to obtain the PySINDy regressed equations we need to provide data of a trajectory predicted by the ASRNN.
- This is provided in the code by specififying a file path, we can use the example data in the repository or additional data in the Supplementary Material, or choose to generate new data using the repository and provide its file path in the next cell.

In [46]:
######### Input file path to trajectory data ############

file_path = 'predictions/hamiltonian_mlp_predictions.npz'

In [48]:
data = np.load(file_path)
params = data['params']
ps_ground = data['ps_ground']
qs_ground = data['qs_ground']
Ks_ground = data['Ks_ground']
Vs_ground = data['Vs_ground']
ps_pred = data['ps_pred']
qs_pred = data['qs_pred']
Ks_pred = data['Ks_pred']
Vs_pred = data['Vs_pred']

In [50]:
t = np.linspace(0, 49.9, 500)
index = 0
qx = qs_pred[:, index, 0]
qy = qs_pred[:, index, 1]
px = ps_pred[:, index, 0]
py = ps_pred[:, index, 1]
X = np.stack((qx, qy, px, py), axis=-1)
print(X.shape)
print(t.shape)
print('True parameters =', params[index])

(500, 4)
(500,)
True parameters = [0.5 0.7]


In [52]:
differentiation_method = ps.FiniteDifference(order=2)
feature_library = ps.PolynomialLibrary(degree=3) #Note that cubic terms (higher order than expected) are also among the basis terms to ensure robust predictions
optimizer = ps.STLSQ(threshold=0.2)

model = ps.SINDy(
    differentiation_method=differentiation_method,
    feature_library=feature_library,
    optimizer=optimizer,
    feature_names=["q1", "q2", "p1", "p2"],
)


In [54]:
model.fit(X, t=t)

SINDy(differentiation_method=FiniteDifference(),
      feature_library=PolynomialLibrary(degree=3),
      feature_names=['q1', 'q2', 'p1', 'p2'], optimizer=STLSQ(threshold=0.2))

### Output

In [57]:
model.print()

(q1)' = 0.998 p1
(q2)' = 0.997 p2
(p1)' = -0.995 q1 + -0.986 q1 q2
(p2)' = -1.000 q2 + -0.511 q1^2 + 0.703 q2^2
